In [2]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell
# Local VS Code users: SKIP this cell
# ============================================================

# !pip install crewai crewai-tools duckduckgo-search python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# print("Colab setup complete!")

# Day 6: Multi-Agent Systems with CrewAI

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Understand why multi-agent beats single agent
2. Learn CrewAI's 3 core concepts: Agent, Task, Crew
3. Build a 3-agent content creation crew (Researcher → Writer → Editor)
4. Add tools to agents
5. Compare Sequential vs Hierarchical process
6. Read and debug verbose crew output

**Deliverable:** 3-agent blog creation crew

**New framework:** CrewAI (multi-agent) — different from LangChain (single-agent pipelines)

---

## 0. Setup

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'Loaded' if len(groq_key) > 10 else 'MISSING'}")

Groq API Key: Loaded


In [4]:
# CrewAI uses its own LLM class (not LangChain's ChatGroq)
from crewai import LLM

# provider/model format — same as Smolagents' LiteLLMModel
llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

print(f"LLM configured: {llm.model}")

LLM configured: groq/llama-3.3-70b-versatile


---
## 1. Why Multiple Agents?

In Days 4-5, you built powerful single agents. But single agents hit a ceiling:

| Problem | What happens |
|---|---|
| **Role confusion** | Agent mixes up steps — writes before researching, skips editing |
| **Prompt overload** | System prompt gets huge with all roles + tools → LLM degrades |
| **No specialization** | Researcher needs to be thorough, writer creative, editor critical — one agent can't be all |

**Solution:** Give each role to a separate agent. Each one focuses on their task.

Like a movie production: screenwriter + director + editor. Not one person doing everything.

---

## 2. CrewAI's 3 Core Concepts

- **Agent (WHO)** — A persona with role, goal, and backstory
- **Task (WHAT)** — A specific job assigned to one agent
- **Crew (HOW)** — The team that coordinates agents and tasks

### 2.1 Creating Agents — Role, Goal, Backstory

In [5]:
from crewai import Agent

# Agent 1: Researcher
researcher = Agent(
    role="Senior Research Analyst",
    goal="Find comprehensive, accurate, and up-to-date information on any given topic",
    backstory=(
        "You are an experienced research analyst who has spent 10 years at top consulting firms. "
        "You are known for your thorough research methodology, attention to detail, and ability "
        "to identify the most relevant and credible sources. You always verify facts from multiple "
        "sources before including them in your reports."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,  # keep it simple — no delegating to other agents
)

# Agent 2: Writer
writer = Agent(
    role="Creative Blog Writer",
    goal="Write engaging, informative blog posts that captivate readers",
    backstory=(
        "You are a talented content writer with a knack for making complex topics accessible "
        "and interesting. Your blog posts have gone viral multiple times. You write in a "
        "conversational yet authoritative tone, using analogies and examples that resonate "
        "with a broad audience. You always structure posts with clear headings and a strong hook."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Agent 3: Editor
editor = Agent(
    role="Grammar and Style Editor",
    goal="Polish content to publication quality with perfect grammar, clarity, and engaging style",
    backstory=(
        "You are a meticulous editor who has worked at leading publications like The Hindu and "
        "India Today. You have an eye for grammatical errors, awkward phrasing, and unclear "
        "sentences. You improve readability while preserving the writer's voice. You also check "
        "for factual consistency and logical flow."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("3 agents created!")
print(f"  1. {researcher.role}")
print(f"  2. {writer.role}")
print(f"  3. {editor.role}")

3 agents created!
  1. Senior Research Analyst
  2. Creative Blog Writer
  3. Grammar and Style Editor


**Why backstory matters:** It's NOT just flavor text. A researcher backstory of "10 years at consulting firms" makes the LLM write formal, data-driven reports. A writer backstory of "viral blog posts" makes it write engaging, accessible content. The backstory fundamentally shapes the output.

### 2.2 Creating Tasks — Description, Expected Output, Agent

In [15]:
from crewai import Task

# Task 1: Research
research_task = Task(
    description=(
        "Research the topic: {topic}. "
        "Find 5 key facts, recent statistics, notable expert opinions, "
        "and any recent developments. Focus on information from 2025-2026. "
        "Verify facts from multiple sources where possible."
    ),
    expected_output=(
        "A structured research report with: "
        "1) An overview paragraph, "
        "2) 5 key findings (each with supporting data), "
        "3) A section on recent trends or developments."
    ),
    agent=researcher,
)

# Task 2: Write
write_task = Task(
    description=(
        "Using the research report provided, write an engaging 400-500 word blog post. "
        "Start with a compelling hook that grabs the reader's attention. "
        "Use 3-4 clear sections with headings. "
        "Include analogies or examples to explain complex concepts. "
        "End with a thought-provoking conclusion."
    ),
    expected_output=(
        "A 400-500 word blog post with: "
        "an engaging title, a hook opening, 3-4 sections with headings, "
        "and a conclusion. Written in a conversational yet authoritative tone."
    ),
    agent=writer,
)

# Task 3: Edit
edit_task = Task(
    description=(
        "Edit the blog post for: "
        "1) Grammar and spelling errors, "
        "2) Awkward or unclear phrasing, "
        "3) Logical flow between sections, "
        "4) Consistency in tone and style, "
        "5) Factual accuracy (cross-check with the research report). "
        "Make improvements while preserving the writer's voice."
    ),
    expected_output=(
        "The final polished blog post, ready for publication. "
        "All grammar issues fixed, unclear sentences rewritten, "
        "and flow improved. Include a brief editor's note listing the changes made."
    ),
    agent=editor,
)

print("3 tasks created!")
print(f"  1. Research → assigned to: {research_task.agent.role}")
print(f"  2. Write   → assigned to: {write_task.agent.role}")
print(f"  3. Edit    → assigned to: {edit_task.agent.role}")

3 tasks created!
  1. Research → assigned to: Senior Research Analyst
  2. Write   → assigned to: Creative Blog Writer
  3. Edit    → assigned to: Grammar and Style Editor


**Notice:**
- Each task has a detailed `description` telling the agent exactly what to do
- `expected_output` sets the quality bar — the agent knows what a good result looks like
- Each task is assigned to exactly ONE agent
- The `{topic}` variable in the research task will be filled when we run the crew

---

## 3. The Crew — Assembling the Team

The Crew coordinates the agents and tasks. We'll use `Process.sequential` — tasks run one after another, each receiving the previous task's output as context.

In [16]:
from crewai import Crew, Process

blog_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, write_task, edit_task],
    process=Process.sequential,  # tasks run in order
    verbose=True,                # see every step
)

print(f"Crew assembled!")
print(f"  Agents: {len(blog_crew.agents)}")
print(f"  Tasks: {len(blog_crew.tasks)}")
print(f"  Process: {blog_crew.process}")

Crew assembled!
  Agents: 3
  Tasks: 3
  Process: Process.sequential


---
## 4. Running the Crew — `crew.kickoff()`

This is where the magic happens. The crew runs all tasks in sequence:
1. Researcher finds facts about the topic
2. Writer receives the research report and creates a blog post
3. Editor receives the draft and polishes it

**Warning:** This makes 3+ LLM calls and uses ~10,000-15,000 tokens. On Groq's free tier, be mindful of rate limits.

In [17]:
# Run the crew!
result = blog_crew.kickoff(inputs={"topic": "AI Agents in India — trends and opportunities in 2026"})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.10.1                                                                                        │
│  Latest version:  1.12.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b2e7f472-bba3-4e8c-98b2-fd341bc66928                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: AI Agents in India — trends and opportunities in 2026. Find 5 key facts, recent      │
│  statistics, notable expert opinions, and any recent developments. Focus on information from 2025-2026. Verify  │
│  facts from multiple sources where possible.                                                                    │
│  ID: 8224e30b-24b0-482e-a7e6-5e9344db1e08                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the topic: AI Agents in India — trends and opportunities in 2026. Find 5 key facts, recent      │
│  statistics, notable expert opinions, and any recent developments. Focus on information from 2025-2026. Verify  │
│  facts from multiple sources where possible.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Overview**                                                                                                   │
│  The integration of Artificial Intelligence (AI) agents in India has witnessed significant growth and           │
│  transformation, especially in the last couple of years. As AI technology continues to advance, its             │
│  application across various sectors in India is becoming more prevalent, driven by the increasing demand for    │
│  automation, personalized customer experiences, and data-driven insights. The year 2026 is poised to be a       │
│  critical year for AI agents in India, with anticipated advancements in AI adoption across industries, the      │
│  emergence of new trends, and the creation of fresh opportunities. This report delves into the current          │
│  landscape of AI agents in India, highlighting key trends, opportunities, and recent developments that are      │
│  shaping the sector.                                                                                            │
│                                                                                                                 │
│  **5 Key Findings**                                                                                             │
│                                                                                                                 │
│  1. **Growing Demand for AI Talent**: According to a report by LinkedIn (2025), the demand for AI               │
│  professionals in India has seen a significant surge, with a 45% increase in job postings in the AI sector      │
│  from 2024 to 2025. This growth is attributed to the increasing adoption of AI technologies by Indian           │
│  businesses, driving the need for skilled professionals who can develop and implement AI solutions. (Source:    │
│  LinkedIn, "2025 LinkedIn Jobs on the Rise Report")                                                             │
│                                                                                                                 │
│  2. **AI Adoption in Healthcare**: A survey conducted by PwC India (2025) found that 71% of healthcare          │
│  providers in India are planning to invest in AI over the next two years to improve patient outcomes, enhance   │
│  operational efficiency, and reduce costs. This high rate of intended adoption underscores the potential of AI  │
│  to revolutionize the healthcare sector in India. (Source: PwC India, "Healthcare in India: A PwC Report")      │
│                                                                                                                 │
│  3. **AI in Customer Service**: A study by Accenture (2025) revealed that 85% of Indian consumers prefer        │
│  interacting with human customer support agents, but 65% are open to using AI-powered chatbots for simple       │
│  queries. This indicates that while AI has a role to play in customer service, human interaction remains        │
│  paramount for more complex issues. (Source: Accenture, "2025 Global Consumer Pulse Research")                  │
│                                                                                                                 │
│  4. **Investment in AI Startups**: India witnessed a considerable increase in investments in AI startups in     │
│  2025, with total funding reaching $1.2 billion, up fro

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Grammar and Style Editor                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Overview**                                                                                                   │
│  The integration of Artificial Intelligence (AI) agents in India has witnessed significant growth and           │
│  transformation over the past couple of years. As AI technology continues to advance, its application across    │
│  various sectors in India is becoming increasingly prevalent, driven by the rising demand for automation,       │
│  personalized customer experiences, and data-driven insights. The year 2026 is poised to be a pivotal year for  │
│  AI agents in India, with anticipated advancements in AI adoption across industries, the emergence of new       │
│  trends, and the creation of fresh opportunities. This report delves into the current landscape of AI agents    │
│  in India, highlighting key trends, opportunities, and recent developments that are shaping the sector.         │
│                                                                                                                 │
│  **5 Key Findings**                                                                                             │
│                                                                                                                 │
│  1. **Growing Demand for AI Talent**: According to a report by LinkedIn (2025), the demand for AI               │
│  professionals in India has surged significantly, with a 45% increase in job postings in the AI sector from     │
│  2024 to 2025. This growth can be attributed to the increasing adoption of AI technologies by Indian            │
│  businesses, which in turn drives the need for skilled professionals who can develop and implement AI           │
│  solutions. (Source: LinkedIn, "2025 LinkedIn Jobs on the Rise Report")                                         │
│                                                                                                                 │
│  2. **AI Adoption in Healthcare**: A survey conducted by PwC India (2025) found that 71% of healthcare          │
│  providers in India plan to invest in AI over the next two years to improve patient outcomes, enhance           │
│  operational efficiency, and reduce costs. This high rate of intended adoption underscores the potential of AI  │
│  to revolutionize the healthcare sector in India. (Source: PwC India, "Healthcare in India: A PwC Report")      │
│                                                                                                                 │
│  3. **AI in Customer Service**: A study by Accenture (2025) revealed that 85% of Indian consumers prefer        │
│  interacting with human customer support agents, but 65% are open to using AI-powered chatbots for simple       │
│  queries. This indicates that while AI has a role to play in customer service, human interaction remains        │
│  paramount for more complex issues. (Source: Accenture, "2025 Global Consumer Pulse Research")                  │
│                                                                                                                 │
│  4. **Investment in AI Startups**: India witnessed a significant increase in investments in AI startups in      │
│  2025, with total funding reaching $1.2 billion, up from $750 million in 2024, according to a report by KPMG.   │
│  This rise in funding is a testament to the growing int

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Edit the blog post for: 1) Grammar and spelling errors, 2) Awkward or unclear phrasing, 3) Logical flow  │
│  between sections, 4) Consistency in tone and style, 5) Factual accuracy (cross-check with the research         │
│  report). Make improvements while preserving the writer's voice.                                                │
│  Agent: Grammar and Style Editor                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b2e7f472-bba3-4e8c-98b2-fd341bc66928                                                                       │
│  Final Output: **Overview**                                                                                     │
│  The integration of Artificial Intelligence (AI) agents in India has witnessed significant growth and           │
│  transformation over the past couple of years. As AI technology continues to advance, its application across    │
│  various sectors in India is becoming increasingly prevalent, driven by the rising demand for automation,       │
│  personalized customer experiences, and data-driven insights. The year 2026 is poised to be a pivotal year for  │
│  AI agents in India, with anticipated advancements in AI adoption across industries, the emergence of new       │
│  trends, and the creation of fresh opportunities. This report delves into the current landscape of AI agents    │
│  in India, highlighting key trends, opportunities, and recent developments that are shaping the sector.         │
│                                                                                                                 │
│  **5 Key Findings**                                                                                             │
│                                                                                                                 │
│  1. **Growing Demand for AI Talent**: According to a report by LinkedIn (2025), the demand for AI               │
│  professionals in India has surged significantly, with a 45% increase in job postings in the AI sector from     │
│  2024 to 2025. This growth can be attributed to the increasing adoption of AI technologies by Indian            │
│  businesses, which in turn drives the need for skilled professionals who can develop and implement AI           │
│  solutions. (Source: LinkedIn, "2025 LinkedIn Jobs on the Rise Report")                                         │
│                                                                                                                 │
│  2. **AI Adoption in Healthcare**: A survey conducted by PwC India (2025) found that 71% of healthcare          │
│  providers in India plan to invest in AI over the next two years to improve patient outcomes, enhance           │
│  operational efficiency, and reduce costs. This high rate of intended adoption underscores the potential of AI  │
│  to revolutionize the healthcare sector in India. (Source: PwC India, "Healthcare in India: A PwC Report")      │
│                                                                                                                 │
│  3. **AI in Customer Service**: A study by Accenture (2025) revealed that 85% of Indian consumers prefer        │
│  interacting with human customer support agents, but 65% are open to using AI-powered chatbots for simple       │
│  queries. This indicates that while AI has a role to play in customer service, human interaction remains        │
│  paramount for more complex issues. (Source: Accenture, "2025 Global Consumer Pulse Research")                  │
│                                                                                                                 │
│  4. **Investment in AI Startups**: India witnessed a significant increase in investments in AI startups in      │
│  2025, with total funding reaching $1.2 billion, up from $750 million in 2024, according to a report by KPMG.   │
│  This rise in funding is a testament to the growing in

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT
**Overview**
The integration of Artificial Intelligence (AI) agents in India has witnessed significant growth and transformation over the past couple of years. As AI technology continues to advance, its application across various sectors in India is becoming increasingly prevalent, driven by the rising demand for automation, personalized customer experiences, and data-driven insights. The year 2026 is poised to be a pivotal year for AI agents in India, with anticipated advancements in AI adoption across industries, the emergence of new trends, and the creation of fresh opportunities. This report delves into the current landscape of AI agents in India, highlighting key trends, opportunities, and recent developments that are shaping the sector.

**5 Key Findings**

1. **Growing Demand for AI Talent**: According to a report by LinkedIn (2025), the demand for AI professionals in India has surged significantly, with a 45% increase in job postings in the AI sector from 2024 to 

**Look at the verbose output above.** You should see:
1. `[Agent: Senior Research Analyst]` working on the research task
2. `[Agent: Creative Blog Writer]` receiving the research and writing the post
3. `[Agent: Grammar and Style Editor]` polishing the final version

Each agent only sees what they need — the researcher's output flows to the writer, the writer's draft flows to the editor.

---

## 5. Adding Tools to Agents

Right now our agents only use their LLM knowledge. Let's give the researcher actual web search capability.

In [9]:
from crewai.tools import tool

@tool("Web Search")
def search_web(query: str) -> str:
    """Searches the web for current information using DuckDuckGo.
    Use this to find recent news, statistics, and expert opinions.
    
    Args:
        query: The search query, e.g. 'AI agents India 2026'
    """
    try:
        from duckduckgo_search import DDGS
        results = DDGS().text(query, max_results=3)
        if not results:
            return f"No results found for '{query}'. Try different keywords."
        output = []
        for i, r in enumerate(results, 1):
            output.append(f"{i}. {r['title']}: {r['body'][:200]}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Search error: {e}. Try a simpler query."

# Test the tool
print(search_web.run("AI agents India 2026")[:300], "...")

Search error: No module named 'duckduckgo_search'. Try a simpler query. ...


In [10]:
# Recreate the researcher with the search tool
researcher_with_tools = Agent(
    role="Senior Research Analyst",
    goal="Find comprehensive, accurate, and up-to-date information using web search",
    backstory=(
        "You are an experienced research analyst with 10 years at top consulting firms. "
        "You use web search to find the latest data and verify facts from multiple sources."
    ),
    tools=[search_web],  # give the researcher web search!
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Update the research task to use the new agent
research_task_v2 = Task(
    description=(
        "Research the topic: {topic}. "
        "Use the web search tool to find 5 key facts with recent data. "
        "Search for statistics, expert opinions, and recent developments."
    ),
    expected_output=(
        "A research report with 5 key findings supported by data from web search results."
    ),
    agent=researcher_with_tools,
)

# Create the enhanced crew
enhanced_crew = Crew(
    agents=[researcher_with_tools, writer, editor],
    tasks=[research_task_v2, write_task, edit_task],
    process=Process.sequential,
    verbose=True,
)

print(f"Enhanced crew ready! Researcher now has: {[t.name for t in researcher_with_tools.tools]}")

Enhanced crew ready! Researcher now has: ['Web Search']


In [11]:
# Run the enhanced crew — researcher will actually search the web
result_v2 = enhanced_crew.kickoff(inputs={"topic": "The rise of AI startups in Hyderabad"})

print("\n" + "=" * 60)
print("FINAL OUTPUT (with web search)")
print("=" * 60)
print(result_v2.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.10.1                                                                                        │
│  Latest version:  1.12.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f0616a8e-a55a-4a0d-bc00-3d14f91b791d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: The rise of AI startups in Hyderabad. Use the web search tool to find 5 key facts    │
│  with recent data. Search for statistics, expert opinions, and recent developments.                             │
│  ID: bececeb9-9cb7-4c94-b211-75ffc53283ba                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the topic: The rise of AI startups in Hyderabad. Use the web search tool to find 5 key facts    │
│  with recent data. Search for statistics, expert opinions, and recent developments.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'AI startups in Hyderabad recent data'}                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Edit the blog post for: 1) Grammar and spelling errors, 2) Awkward or unclear phrasing, 3) Logical flow  │
│  between sections, 4) Consistency in tone and style, 5) Factual accuracy (cross-check with the research         │
│  report). Make improvements while preserving the writer's voice.                                                │
│  Agent: Grammar and Style Editor                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: f0616a8e-a55a-4a0d-bc00-3d14f91b791d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kfzzwk76f0y83hedfy0vefyv` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 8121, Requested 5180. Please try again in 6.505s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


**Compare the two runs:**
- Without tools: researcher uses only its training data (may be outdated)
- With tools: researcher actively searches the web for current information

The writer and editor don't need tools — they work with the text the researcher provides.

**Tip:** Only give each agent the tools they actually need. More tools = more token cost (tool descriptions are sent with every LLM call).

---

## 6. Inspecting Results — Token Usage and Task Outputs

CrewAI provides metadata about the run:

In [ ]:
# Inspect the result object
print("=== Result Metadata ===")
print(f"Type: {type(result_v2).__name__}")
print()

# Token usage (if available)
if hasattr(result_v2, 'token_usage'):
    print(f"Token usage: {result_v2.token_usage}")

# Individual task outputs
if hasattr(result_v2, 'tasks_output'):
    print("\n=== Task Outputs ===")
    for i, task_output in enumerate(result_v2.tasks_output):
        agent_name = [researcher_with_tools.role, writer.role, editor.role][i]
        print(f"\n--- Task {i+1}: {agent_name} ---")
        print(str(task_output)[:300], "...")

=== Result Metadata ===
Type: CrewOutput

Token usage: total_tokens=46806 prompt_tokens=32526 cached_prompt_tokens=0 completion_tokens=14280 successful_requests=21

=== Task Outputs ===

--- Task 1: Senior Research Analyst ---
The rise of AI startups in Hyderabad has gained significant momentum in recent years. Here are 5 key findings based on recent data and expert opinions:

1. **Growing Number of AI Startups**: As of 2022, there are over 200 AI startups in Hyderabad, with many more in the pipeline. This number has been ...

--- Task 2: Creative Blog Writer ---
**The AI Revolution in Hyderabad: How the City is Emerging as a Hub for Artificial Intelligence**

Imagine a city where machines can think, learn, and act like humans. Sounds like science fiction, right? But, this is the reality in Hyderabad, where the rise of AI startups has gained significant mome ...

--- Task 3: Grammar and Style Editor ---
**The AI Revolution in Hyderabad: How the City is Emerging as a Hub for Artificial 

: 

---
## 7. Different Topic — Same Crew

The crew is reusable! Just change the `inputs` to generate content on any topic.

In [ ]:
# Same crew, different topic
result_v3 = enhanced_crew.kickoff(inputs={"topic": "How B.Tech students can build a career in AI engineering"})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result_v3.raw)

NameError: name 'enhanced_crew' is not defined

---
## 8. Sequential vs Hierarchical — Quick Comparison

We've been using `Process.sequential` (tasks run in order). CrewAI also supports `Process.hierarchical` where a manager agent coordinates the team.

| | Sequential | Hierarchical |
|---|---|---|
| **Flow** | Task 1 → Task 2 → Task 3 | Manager delegates to agents |
| **Control** | You define the order | Manager decides the order |
| **Token cost** | Lower (3 LLM calls) | Higher (manager adds extra calls) |
| **Best for** | Clear dependency chains | Complex tasks with unclear ordering |
| **Our choice** | This course uses sequential | Explore on your own |

For most use cases, sequential is simpler, cheaper, and more predictable.

---

## 9. Your Turn — Build a Custom Crew

**Challenge (10 minutes):** Build a crew for a different domain.

**Ideas:**
- **Product Review Crew:** Reviewer (search for reviews) → Summarizer → Recommendation Writer
- **Study Notes Crew:** Researcher → Note Taker → Quiz Generator
- **Travel Planner Crew:** Destination Researcher → Itinerary Planner → Budget Calculator
- **Code Review Crew:** Code Analyzer → Bug Finder → Improvement Suggester

**Requirements:**
1. At least 2 agents with distinct roles/backstories
2. At least 2 tasks with clear expected outputs
3. Sequential process
4. Test with at least 1 topic

In [ ]:
# YOUR CODE HERE

# Step 1: Create your agents
# agent1 = Agent(
#     role="...",
#     goal="...",
#     backstory="...",
#     llm=llm,
#     verbose=True,
#     allow_delegation=False,
# )

# Step 2: Create your tasks
# task1 = Task(
#     description="...",
#     expected_output="...",
#     agent=agent1,
# )

# Step 3: Create and run the crew
# my_crew = Crew(
#     agents=[agent1, agent2],
#     tasks=[task1, task2],
#     process=Process.sequential,
#     verbose=True,
# )
# result = my_crew.kickoff(inputs={"topic": "your topic here"})
# print(result.raw)

---
## 10. Framework Comparison — Your Journey So Far

| | Smolagents (Days 1-2) | LangChain (Days 4-5) | CrewAI (Day 6) |
|---|---|---|---|
| **Built by** | HuggingFace | LangChain Inc. | CrewAI Inc. |
| **Best for** | Quick prototyping | Production pipelines | Multi-agent teams |
| **Agents** | Single agent | Single agent | Multiple agents! |
| **Complexity** | 3 lines | 10-15 lines | 20-30 lines |
| **Key concept** | `CodeAgent` | `AgentExecutor` | `Crew + Process` |
| **LLM class** | `LiteLLMModel("groq/...")` | `ChatGroq("...")` | `LLM("groq/...")` |

**Progression:** Smolagents (learn fast) → LangChain (build pipelines) → CrewAI (multi-agent) → **LangGraph (tomorrow!)**

---
## 11. Day 6 Wrap-up

### What you built today:
- Understood why multi-agent beats single agent (role confusion, prompt overload, no specialization)
- Learned CrewAI's 3 core concepts: Agent (WHO), Task (WHAT), Crew (HOW)
- Built a 3-agent blog crew: Researcher → Writer → Editor
- Added web search tool to the researcher
- Compared Sequential vs Hierarchical process
- Inspected token usage and task outputs

### Key CrewAI pattern:
```python
agent = Agent(role=..., goal=..., backstory=..., tools=[...], llm=llm)
task  = Task(description=..., expected_output=..., agent=agent)
crew  = Crew(agents=[...], tasks=[...], process=Process.sequential)
result = crew.kickoff(inputs={"topic": "..."})
```

### Token cost reminder:
- Single LLM call: ~500 tokens (1x)
- Single agent (Day 5): ~3,000 tokens (6x)
- 3-agent crew: ~10,000-15,000 tokens (10-15x)
- Use 8B model for testing, 70B for final runs

### Tomorrow — Day 7: Advanced Multi-Agent Patterns
Custom domain agents, hierarchical process with manager agent, inter-agent communication, multi-agent RAG, and a real-world competitive analysis use case.

---

### Resources:
- [CrewAI docs](https://docs.crewai.com/)
- [CrewAI GitHub](https://github.com/crewAIInc/crewAI)
- [CrewAI examples](https://docs.crewai.com/examples/)